# Module 8 · System Observability Apm

**Track:** AI Engineering Core  
**Objective:** Master the principles and applications of System Observability Apm.

---



## 📑 Table of Contents

* [Why This Notebook Is Non-Negotiable](#why-this-notebook-is-non-negotiable)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Three Pillars of Observability](#1-the-three-pillars-of-observability)
  * [How They Work Together](#how-they-work-together)
  * [MELT vs RED vs USE: Monitoring Frameworks](#melt-vs-red-vs-use-monitoring-frameworks)
* [2 · Metrics: Prometheus for AI Systems](#2-metrics-prometheus-for-ai-systems)
  * [Architecture](#architecture)
  * [Prometheus Metric Types](#prometheus-metric-types)
* [3 · Logs: Structured Logging & the ELK Stack](#3-logs-structured-logging-the-elk-stack)
  * [Unstructured vs Structured Logs](#unstructured-vs-structured-logs)
  * [ELK Stack Architecture](#elk-stack-architecture)
  * [Modern Alternatives](#modern-alternatives)
* [4 · Traces: Distributed Request Tracing](#4-traces-distributed-request-tracing)
* [5 · Dashboards: Grafana for AI Systems](#5-dashboards-grafana-for-ai-systems)
  * [Essential AI System Dashboard Panels](#essential-ai-system-dashboard-panels)
* [6 · Alerting](#6-alerting)
  * [Alert Design Principles](#alert-design-principles)
* [7 · GPU-Specific Monitoring](#7-gpu-specific-monitoring)
  * [DCGM Exporter (NVIDIA Data Center GPU Manager)](#dcgm-exporter-nvidia-data-center-gpu-manager)
* [Knowledge Check](#knowledge-check)
  * [Q1: RED vs USE](#q1-red-vs-use)
  * [Q2: Counter vs Gauge](#q2-counter-vs-gauge)
  * [Q3: Structured Logging](#q3-structured-logging)
  * [Q4: Alert Design](#q4-alert-design)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Instrument an API](#exercise-1-instrument-an-api)
  * [Exercise 2: Design Alerts](#exercise-2-design-alerts)
  * [Exercise 3: Structured Logging](#exercise-3-structured-logging)
* [Summary](#summary)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors debug production issues by SSH-ing into a server and running `top`. Seniors have pre-built Grafana dashboards that show GPU utilization, request latency percentiles, error rates, and pod memory — across 50 nodes simultaneously — with alerts that wake the on-call engineer before users notice.

**Mental Model:** Observability is like the instrument panel in an airplane cockpit. You don't fly by looking out the window (SSH-ing into servers). You fly by reading altimeters, fuel gauges, and engine temperature (Prometheus metrics, Grafana dashboards, structured logs). Without instruments, you're flying blind.

**Common Junior Pitfall:** Using `print()` statements for everything, then searching through unstructured log files with `grep` when production breaks. Structured logging with JSON fields lets you query instantly: "show me all requests where `latency > 5s AND model = llama-3 AND gpu_utilization > 95%`".

---

In [ ]:
# Cell 0 — Install Dependencies
!pip install -q prometheus-client structlog opentelemetry-api opentelemetry-sdk psutil

---
## 1 · The Three Pillars of Observability

| Pillar | What It Captures | Tool | Example |
|--------|-----------------|------|--------|
| **Metrics** | Numeric measurements over time | Prometheus, Datadog | GPU utilization: 87% |
| **Logs** | Discrete events with context | ELK, Loki, CloudWatch | `{"level":"error", "msg":"OOM", "gpu":2}` |
| **Traces** | Request journey through services | Jaeger, Tempo, Zipkin | API → auth → cache → model → response |

### How They Work Together

```
ALERT: p99 latency > 5s
  → Check METRICS dashboard: GPU utilization = 99%, queue depth = 500
  → Check LOGS: "CUDA OOM on gpu:3, falling back to gpu:1"
  → Check TRACES: request stuck 4.8s waiting for GPU allocation
  → ROOT CAUSE: gpu:3 has a leaked tensor from yesterday's training job
```

### MELT vs RED vs USE: Monitoring Frameworks

| Framework | Focus | Metrics |
|-----------|-------|--------|
| **RED** | Request-focused (APIs) | **R**ate, **E**rror rate, **D**uration |
| **USE** | Resource-focused (infra) | **U**tilization, **S**aturation, **E**rrors |
| **MELT** | Full stack | **M**etrics, **E**vents, **L**ogs, **T**races |

In [ ]:
# Cell 1 — Demonstrate the three pillars with a simulated AI system

import time
import json
import random
import uuid
from datetime import datetime

# === PILLAR 1: Metrics (numeric, time-series) ===
print('=== Pillar 1: METRICS (what Prometheus stores) ===')
metrics_data = [
    {'metric': 'gpu_utilization_percent', 'gpu': '0', 'value': 87.3, 'timestamp': '2024-03-15T10:00:00Z'},
    {'metric': 'gpu_memory_used_bytes', 'gpu': '0', 'value': 38_654_705_664, 'timestamp': '2024-03-15T10:00:00Z'},
    {'metric': 'inference_requests_total', 'model': 'llama-3-70b', 'value': 15420, 'timestamp': '2024-03-15T10:00:00Z'},
    {'metric': 'inference_latency_seconds', 'quantile': '0.99', 'value': 4.7, 'timestamp': '2024-03-15T10:00:00Z'},
]
for m in metrics_data:
    print(f'  {m["metric"]}{{{list(m.keys())[1]}="{list(m.values())[1]}"}} {m["value"]}')

# === PILLAR 2: Logs (discrete events) ===
print('\n=== Pillar 2: LOGS (what ELK/Loki stores) ===')
log_entries = [
    {'timestamp': '2024-03-15T10:00:01Z', 'level': 'INFO', 'service': 'inference-api',
     'msg': 'Request completed', 'request_id': 'req-abc123', 'model': 'llama-3-70b',
     'latency_ms': 1240, 'tokens_generated': 156, 'gpu_id': 0},
    {'timestamp': '2024-03-15T10:00:02Z', 'level': 'WARNING', 'service': 'inference-api',
     'msg': 'GPU memory pressure', 'gpu_id': 0, 'used_pct': 94.2,
     'action': 'evicting_cache'},
    {'timestamp': '2024-03-15T10:00:03Z', 'level': 'ERROR', 'service': 'inference-api',
     'msg': 'CUDA OOM', 'gpu_id': 2, 'requested_mb': 4096, 'available_mb': 512,
     'model': 'llama-3-70b', 'request_id': 'req-def456'},
]
for log in log_entries:
    print(f'  {json.dumps(log)}')

# === PILLAR 3: Traces (request journey) ===
print('\n=== Pillar 3: TRACES (what Jaeger stores) ===')
trace_id = 'trace-' + uuid.uuid4().hex[:12]
spans = [
    {'trace_id': trace_id, 'span': 'api.chat_completions', 'duration_ms': 1240, 'status': 'OK'},
    {'trace_id': trace_id, 'span': '  → auth.validate_jwt', 'duration_ms': 2, 'status': 'OK'},
    {'trace_id': trace_id, 'span': '  → cache.check_semantic', 'duration_ms': 8, 'status': 'MISS'},
    {'trace_id': trace_id, 'span': '  → gpu.allocate_slot', 'duration_ms': 45, 'status': 'OK'},
    {'trace_id': trace_id, 'span': '  → model.generate_tokens', 'duration_ms': 1180, 'status': 'OK'},
    {'trace_id': trace_id, 'span': '    → model.prefill', 'duration_ms': 120, 'status': 'OK'},
    {'trace_id': trace_id, 'span': '    → model.decode (156 tokens)', 'duration_ms': 1060, 'status': 'OK'},
]
for span in spans:
    print(f'  {span["span"]:45s} {span["duration_ms"]:6d}ms  {span["status"]}')

---
## 2 · Metrics: Prometheus for AI Systems

Prometheus is the standard for infrastructure metrics. It **pulls** metrics from your services via HTTP.

### Architecture

```
┌──────────────┐     ┌──────────────┐     ┌─────────────┐
│ Your FastAPI  │────→│  Prometheus   │←────│ Grafana     │
│ /metrics      │pull │  (TSDB)       │     │ (dashboard) │
└──────────────┘     │               │     └─────────────┘
                     │  scrapes      │     
┌──────────────┐     │  every 15s    │     ┌─────────────┐
│ DCGM Exporter │────→│               │────→│ Alertmanager│
│ (GPU metrics) │pull └──────────────┘     │ → PagerDuty │
└──────────────┘                           └─────────────┘
```

### Prometheus Metric Types

| Type | Use | Example |
|------|-----|--------|
| **Counter** | Monotonically increasing | `inference_requests_total` |
| **Gauge** | Can go up or down | `gpu_utilization_percent` |
| **Histogram** | Distribution of values | `inference_latency_seconds` |
| **Summary** | Pre-calculated percentiles | `inference_latency_p99` |

In [ ]:
# Cell 2 — Instrument a FastAPI AI service with Prometheus metrics

from prometheus_client import (
    Counter, Gauge, Histogram, Summary,
    generate_latest, CollectorRegistry, CONTENT_TYPE_LATEST
)
import time
import random

# Create a custom registry (avoids conflicts in notebooks)
registry = CollectorRegistry()

# === Define AI-specific metrics ===

# Counter: total requests (only goes UP)
REQUEST_COUNT = Counter(
    'inference_requests_total',
    'Total inference requests',
    ['model', 'status'],  # Labels for filtering
    registry=registry,
)

# Histogram: latency distribution (buckets for percentile calculation)
REQUEST_LATENCY = Histogram(
    'inference_latency_seconds',
    'Inference request latency',
    ['model'],
    buckets=[0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 10.0, 30.0],  # Seconds
    registry=registry,
)

# Gauge: current values (goes up AND down)
GPU_UTILIZATION = Gauge(
    'gpu_utilization_percent',
    'Current GPU utilization',
    ['gpu_id'],
    registry=registry,
)

GPU_MEMORY_USED = Gauge(
    'gpu_memory_used_bytes',
    'GPU memory in use',
    ['gpu_id'],
    registry=registry,
)

ACTIVE_REQUESTS = Gauge(
    'inference_active_requests',
    'Currently processing requests',
    registry=registry,
)

QUEUE_DEPTH = Gauge(
    'inference_queue_depth',
    'Requests waiting in queue',
    registry=registry,
)

TOKENS_GENERATED = Counter(
    'tokens_generated_total',
    'Total tokens generated',
    ['model'],
    registry=registry,
)

# === Simulate AI workload ===
random.seed(42)
models = ['llama-3-70b', 'llama-3-8b', 'mistral-7b']

for _ in range(100):
    model = random.choice(models)
    latency = random.lognormal(0.5, 0.8)  # Realistic latency distribution
    status = 'success' if random.random() > 0.05 else 'error'
    tokens = random.randint(50, 500)
    
    REQUEST_COUNT.labels(model=model, status=status).inc()
    REQUEST_LATENCY.labels(model=model).observe(latency)
    TOKENS_GENERATED.labels(model=model).inc(tokens)

# Set current GPU state
for gpu_id in range(4):
    GPU_UTILIZATION.labels(gpu_id=str(gpu_id)).set(random.uniform(60, 99))
    GPU_MEMORY_USED.labels(gpu_id=str(gpu_id)).set(
        random.randint(30_000_000_000, 48_000_000_000)
    )

ACTIVE_REQUESTS.set(12)
QUEUE_DEPTH.set(3)

# === Display the /metrics endpoint output ===
output = generate_latest(registry).decode('utf-8')
print('=== Prometheus /metrics Endpoint Output (first 60 lines) ===')
for line in output.split('\n')[:60]:
    print(f'  {line}')
print(f'  ... ({len(output.split(chr(10)))} total lines)')

In [ ]:
# Cell 3 — FastAPI middleware for automatic metrics collection

fastapi_metrics_code = '''
from fastapi import FastAPI, Request
from fastapi.responses import Response
from prometheus_client import Counter, Histogram, Gauge, generate_latest
import time

app = FastAPI()

# Metrics
REQUEST_COUNT = Counter("http_requests_total", "Total HTTP requests", ["method", "path", "status"])
REQUEST_LATENCY = Histogram("http_request_duration_seconds", "Request latency", ["method", "path"])
ACTIVE = Gauge("http_requests_in_progress", "In-flight requests")

@app.middleware("http")
async def metrics_middleware(request: Request, call_next):
    """Automatically instrument every endpoint."""
    ACTIVE.inc()
    start = time.perf_counter()
    
    response = await call_next(request)
    
    duration = time.perf_counter() - start
    REQUEST_COUNT.labels(
        method=request.method, path=request.url.path, status=response.status_code
    ).inc()
    REQUEST_LATENCY.labels(method=request.method, path=request.url.path).observe(duration)
    ACTIVE.dec()
    return response

@app.get("/metrics")
async def prometheus_metrics():
    """Prometheus scrapes this endpoint every 15 seconds."""
    return Response(content=generate_latest(), media_type="text/plain")

@app.post("/v1/chat/completions")
async def chat(...):
    # Your inference logic — metrics are collected automatically by middleware
    pass
'''
print('=== FastAPI Prometheus Middleware ===')
print(fastapi_metrics_code)

---
## 3 · Logs: Structured Logging & the ELK Stack

### Unstructured vs Structured Logs

```
UNSTRUCTURED (bad):
  2024-03-15 10:00:01 INFO Request completed in 1.24s for model llama-3
  → How do you query "all requests > 5s for llama-3 on gpu:2"? Regex? 😱

STRUCTURED (good):
  {"ts":"2024-03-15T10:00:01Z","level":"info","msg":"Request completed",
   "latency_s":1.24,"model":"llama-3","gpu":2,"tokens":156}
  → Query: level=info AND latency_s>5 AND model="llama-3" AND gpu=2 ✅
```

### ELK Stack Architecture

```
Your Services          Elasticsearch     Kibana
┌───────────┐         ┌────────────┐    ┌──────────────┐
│ FastAPI    │──JSON──→│            │───→│ Search logs  │
│ Workers    │  logs   │ Index +    │    │ Visualize    │
│ Trainers   │         │ Store      │    │ Alert        │
└───────────┘         └────────────┘    └──────────────┘
     via Filebeat/Fluentd (log shippers)
```

### Modern Alternatives

| Stack | Components | Best For |
|-------|-----------|----------|
| **ELK** | Elasticsearch + Logstash + Kibana | Full-text search, complex queries |
| **PLG** | Promtail + Loki + Grafana | Lightweight, K8s-native, Grafana integration |
| **Cloud** | CloudWatch / GCP Logging | Managed, zero-ops |

In [ ]:
# Cell 4 — Structured logging with structlog

import structlog
import json

# Configure structured JSON logging
structlog.configure(
    processors=[
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.add_log_level,
        structlog.processors.JSONRenderer(),
    ],
    logger_factory=structlog.PrintLoggerFactory(),
)

logger = structlog.get_logger()

# === Demo: AI inference logging ===
print('=== Structured Logs (JSON → queryable in ELK/Loki) ===')
print()

# Bind context that persists across all log calls
request_logger = logger.bind(
    service="inference-api",
    request_id="req-abc123",
    user_id="user_42",
    model="llama-3-70b",
)

request_logger.info("request_received", endpoint="/v1/chat/completions", prompt_tokens=245)
request_logger.info("gpu_allocated", gpu_id=2, queue_wait_ms=45)
request_logger.info("generation_started", max_tokens=512, temperature=0.7)
request_logger.info("request_completed", 
    latency_ms=1240, 
    tokens_generated=156,
    tokens_per_second=125.8,
    gpu_memory_used_pct=87.3,
)

# Error logging with full context
request_logger.error("cuda_oom",
    gpu_id=2,
    requested_mb=4096,
    available_mb=512,
    action="falling_back_to_gpu_1",
)

print('\n--- ELK Query Examples ---')
print('  Find OOM errors:      level:error AND msg:cuda_oom')
print('  Slow requests:        latency_ms:>5000 AND model:llama-3-70b')
print('  GPU 2 issues:         gpu_id:2 AND level:(error OR warning)')
print('  User activity:        user_id:user_42 AND service:inference-api')

---
## 4 · Traces: Distributed Request Tracing

When a request passes through 5 services, you need to trace its journey to find bottlenecks.

In [ ]:
# Cell 5 — OpenTelemetry tracing for AI pipelines

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

# Setup (production: export to Jaeger/Tempo instead of console)
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("ai-inference-service")

import time, random

def simulate_inference_with_tracing():
    """Demonstrates distributed tracing for an AI inference request."""
    
    with tracer.start_as_current_span("chat_completions") as root:
        root.set_attribute("model", "llama-3-70b")
        root.set_attribute("user_id", "user_42")
        
        # Span 1: Authentication
        with tracer.start_as_current_span("validate_jwt") as auth_span:
            time.sleep(0.002)
            auth_span.set_attribute("auth.method", "jwt")
            auth_span.set_attribute("auth.role", "pro")
        
        # Span 2: Cache check
        with tracer.start_as_current_span("semantic_cache_lookup") as cache_span:
            time.sleep(0.008)
            cache_hit = random.random() > 0.7
            cache_span.set_attribute("cache.hit", cache_hit)
        
        if not cache_hit:
            # Span 3: GPU allocation
            with tracer.start_as_current_span("allocate_gpu") as gpu_span:
                time.sleep(0.045)
                gpu_span.set_attribute("gpu.id", 2)
                gpu_span.set_attribute("gpu.queue_wait_ms", 45)
            
            # Span 4: Model inference
            with tracer.start_as_current_span("model_generate") as gen_span:
                time.sleep(0.1)
                gen_span.set_attribute("tokens.prompt", 245)
                gen_span.set_attribute("tokens.generated", 156)
                gen_span.set_attribute("tokens_per_second", 125.8)

print('=== OpenTelemetry Trace (simplified output) ===')
simulate_inference_with_tracing()
print('\nIn production, these traces appear in Jaeger/Grafana Tempo')
print('as a waterfall diagram showing where time is spent.')

---
## 5 · Dashboards: Grafana for AI Systems

### Essential AI System Dashboard Panels

| Panel | Metric | Alert Threshold |
|-------|--------|----------------|
| **Request Rate** | `rate(inference_requests_total[5m])` | < 10 req/s (traffic drop) |
| **Error Rate** | `rate(requests{status=~"5.."}[5m]) / rate(requests_total[5m])` | > 1% |
| **p99 Latency** | `histogram_quantile(0.99, inference_latency_seconds)` | > 5s |
| **GPU Utilization** | `gpu_utilization_percent` | > 95% sustained |
| **GPU Memory** | `gpu_memory_used_bytes / gpu_memory_total_bytes` | > 90% |
| **Queue Depth** | `inference_queue_depth` | > 50 (requests backing up) |
| **Pod Memory** | `container_memory_usage_bytes` | > 85% of limit |
| **Token Throughput** | `rate(tokens_generated_total[5m])` | Trend monitoring |

In [ ]:
# Cell 6 — Grafana dashboard JSON (importable)

grafana_dashboard = {
    "title": "AI Inference System",
    "panels": [
        {
            "title": "Request Rate (req/s)",
            "type": "timeseries",
            "query": 'rate(inference_requests_total{status="success"}[5m])',
        },
        {
            "title": "Error Rate (%)",
            "type": "gauge",
            "query": '100 * rate(inference_requests_total{status="error"}[5m]) / rate(inference_requests_total[5m])',
            "thresholds": [{"value": 1, "color": "yellow"}, {"value": 5, "color": "red"}],
        },
        {
            "title": "p50 / p95 / p99 Latency",
            "type": "timeseries",
            "queries": [
                'histogram_quantile(0.50, rate(inference_latency_seconds_bucket[5m]))',
                'histogram_quantile(0.95, rate(inference_latency_seconds_bucket[5m]))',
                'histogram_quantile(0.99, rate(inference_latency_seconds_bucket[5m]))',
            ],
        },
        {
            "title": "GPU Utilization by Device",
            "type": "timeseries",
            "query": 'gpu_utilization_percent',
            "legend": '{{gpu_id}}',
        },
        {
            "title": "GPU Memory Usage",
            "type": "bar_gauge",
            "query": 'gpu_memory_used_bytes / gpu_memory_total_bytes * 100',
            "thresholds": [{"value": 80, "color": "yellow"}, {"value": 95, "color": "red"}],
        },
        {
            "title": "Queue Depth",
            "type": "stat",
            "query": 'inference_queue_depth',
        },
    ]
}

print('=== Grafana Dashboard Definition ===')
print(json.dumps(grafana_dashboard, indent=2))
print('\nImport this JSON into Grafana → Dashboards → Import')
print('Connect Prometheus as data source at http://prometheus:9090')

---
## 6 · Alerting

### Alert Design Principles

| Principle | Bad | Good |
|-----------|-----|------|
| Alert on symptoms, not causes | "CPU > 90%" | "p99 latency > 5s" |
| Include runbook link | "Error rate high" | "Error rate > 5%. See: runbook.md#high-error-rate" |
| Avoid alert fatigue | 50 alerts/day | 2-3 actionable alerts/week |

In [ ]:
# Cell 7 — Prometheus alerting rules

alerting_rules = '''
# prometheus-alerts.yml
groups:
  - name: ai_inference_alerts
    rules:
      - alert: HighErrorRate
        expr: |
          rate(inference_requests_total{status="error"}[5m])
          / rate(inference_requests_total[5m]) > 0.05
        for: 5m           # Must persist for 5 min (avoid flapping)
        labels:
          severity: critical
        annotations:
          summary: "Error rate > 5% for {{ $labels.model }}"
          runbook: "https://wiki.company.com/runbooks/high-error-rate"

      - alert: HighP99Latency
        expr: |
          histogram_quantile(0.99, rate(inference_latency_seconds_bucket[5m])) > 10
        for: 10m
        labels:
          severity: warning
        annotations:
          summary: "p99 latency > 10s for {{ $labels.model }}"

      - alert: GPUMemoryCritical
        expr: gpu_memory_used_bytes / gpu_memory_total_bytes > 0.95
        for: 2m
        labels:
          severity: critical
        annotations:
          summary: "GPU {{ $labels.gpu_id }}: memory > 95%. OOM imminent."

      - alert: QueueBacklog
        expr: inference_queue_depth > 100
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "Request queue depth {{ $value }} — scale up workers"
'''
print('=== Prometheus Alerting Rules ===')
print(alerting_rules)

---
## 7 · GPU-Specific Monitoring

### DCGM Exporter (NVIDIA Data Center GPU Manager)

```bash
# Deploy DCGM exporter in K8s (exposes GPU metrics to Prometheus)
helm install dcgm-exporter gpu-helm-charts/dcgm-exporter

# Key metrics exposed:
# DCGM_FI_DEV_GPU_UTIL        → GPU compute utilization %
# DCGM_FI_DEV_FB_USED         → Framebuffer (VRAM) used in MB
# DCGM_FI_DEV_GPU_TEMP        → GPU temperature (throttle at 83°C)
# DCGM_FI_DEV_POWER_USAGE     → Power draw in watts
# DCGM_FI_DEV_SM_CLOCK        → Streaming multiprocessor clock speed
# DCGM_FI_DEV_PCIE_TX_THROUGHPUT → PCIe bandwidth (data transfer bottleneck)
```

| Metric | Warning | Critical | Action |
|--------|---------|----------|--------|
| GPU Temp | > 80°C | > 85°C | Check cooling, reduce batch size |
| VRAM Used | > 85% | > 95% | Reduce KV cache, use quantization |
| GPU Util | < 30% | Sustained | Right-size GPU type (waste!) |
| PCIe TX | > 80% bandwidth | Sustained | Data loading bottleneck |

In [ ]:
# Cell 8 — System metrics collection with psutil

import psutil
import os

def collect_system_metrics():
    """Collect host-level metrics (what node_exporter does for Prometheus)."""
    cpu = psutil.cpu_percent(interval=0.5, percpu=True)
    mem = psutil.virtual_memory()
    disk = psutil.disk_usage('/')
    net = psutil.net_io_counters()
    
    metrics = {
        'cpu': {
            'avg_percent': sum(cpu) / len(cpu),
            'per_core': cpu,
            'core_count': len(cpu),
        },
        'memory': {
            'total_gb': round(mem.total / 1e9, 1),
            'used_gb': round(mem.used / 1e9, 1),
            'percent': mem.percent,
        },
        'disk': {
            'total_gb': round(disk.total / 1e9, 1),
            'used_percent': disk.percent,
        },
        'network': {
            'bytes_sent_mb': round(net.bytes_sent / 1e6, 1),
            'bytes_recv_mb': round(net.bytes_recv / 1e6, 1),
        },
    }
    return metrics

metrics = collect_system_metrics()
print('=== Host System Metrics ===')
print(json.dumps(metrics, indent=2))

# GPU metrics (requires nvidia-smi)
print('\n=== GPU Metrics (nvidia-smi command) ===')
print('''
# Quick check:
nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu \
           --format=csv,noheader

# Output:
# 0, NVIDIA A100-SXM4-80GB, 87 %, 38654 MiB, 81920 MiB, 72
# 1, NVIDIA A100-SXM4-80GB, 92 %, 42100 MiB, 81920 MiB, 75

# Continuous monitoring (1 sample/sec):
nvidia-smi dmon -s u -d 1
''')

---
## Knowledge Check

### Q1: RED vs USE
Your API's p99 latency spiked to 15s. Should you start with RED or USE analysis?

<details><summary>Answer</summary>

Start with **RED** (Rate, Error rate, Duration) — the symptom is latency, which is the D in RED. Check: has throughput increased? Is the error rate up? Then drill into **USE** to find the resource bottleneck: is the GPU saturated? Is memory full? Is the network congested?
</details>

### Q2: Counter vs Gauge
Which metric type for GPU temperature? Which for total requests served?

<details><summary>Answer</summary>

GPU temperature = **Gauge** (goes up and down). Total requests = **Counter** (only increases). Key difference: you apply `rate()` to counters to get per-second values (`requests per second`), but read gauges directly (`current temperature`).
</details>

### Q3: Structured Logging
Why use `{"latency_ms": 1240}` instead of `f"latency: 1240ms"` in production logs?

<details><summary>Answer</summary>

Structured JSON fields are **queryable**: `latency_ms > 5000` in ELK/Loki returns all slow requests instantly. Unstructured strings require regex, which is fragile and slow. Structured logs also enable dashboards (average latency over time) and alerts (latency > threshold).
</details>

### Q4: Alert Design
Why alert on "p99 latency > 5s" instead of "CPU > 90%"?

<details><summary>Answer</summary>

Alert on **symptoms** (user impact), not **causes** (resource metrics). CPU at 90% might be perfectly normal during peak hours. But p99 > 5s means users are having a bad experience regardless of the cause. Investigate the cause (CPU? GPU? network?) after the alert fires.
</details>

---
## Practical Practice

### Exercise 1: Instrument an API
Add Prometheus metrics to a FastAPI inference endpoint. Track: request count by model, latency histogram, active GPU slots, and tokens generated per minute.

### Exercise 2: Design Alerts
Write Prometheus alerting rules for: (1) Error rate > 5% for 5 minutes, (2) No requests for 10 minutes (service down?), (3) GPU temperature > 83°C.

### Exercise 3: Structured Logging
Refactor this unstructured code to use structlog with proper JSON fields:
```python
print(f"Processed request in {elapsed}s, generated {tokens} tokens on GPU {gpu_id}")
if error:
    print(f"ERROR: {error} for user {user_id}")
```

In [ ]:
# ===== EXERCISE SOLUTIONS =====
import structlog
structlog.configure(processors=[structlog.processors.TimeStamper(fmt='iso'),
                                structlog.processors.add_log_level,
                                structlog.processors.JSONRenderer()],
                    logger_factory=structlog.PrintLoggerFactory())

print('=== Exercise 3 Solution: Structured Logging ===')
log = structlog.get_logger().bind(service='inference-api')

# Before (unstructured)
print('BEFORE: print(f"Processed request in {1.24}s...")')
print()

# After (structured)
print('AFTER:')
log.info('request_completed', latency_s=1.24, tokens=156, gpu_id=2, user_id='user_42')
log.error('inference_failed', error='CUDA OOM', gpu_id=2, user_id='user_42',
          requested_mb=4096, available_mb=512)

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Three Pillars | Metrics (Prometheus), Logs (ELK/Loki), Traces (Jaeger/Tempo) |
| RED/USE | RED for request health, USE for resource health |
| Prometheus | Counters, gauges, histograms — /metrics endpoint |
| Structured logging | JSON fields > string formatting — queryable in ELK |
| OpenTelemetry | Distributed tracing across microservices |
| Grafana | Dashboards for GPU util, latency percentiles, queue depth |
| Alerting | Alert on symptoms, include runbooks, avoid fatigue |
| GPU monitoring | DCGM exporter, nvidia-smi, thermal throttling |

**Connections:** Model drift monitoring → NB35 | LLM token observability → NB36 | K8s infrastructure → NB02 | Cost management → NB36  
**Next →** `42_02_monitoring_feedback.ipynb` — Layer 6 — Monitoring & Feedback: Drift Detection & Automated Alerting